# E2 UNet Patch384 + Sliding-Window Full-Image Evaluation

Kaggle notebook for Phase 3: train UNet on `384x384` patches, select `best.pt` by full-image sliding-window validation Dice, then evaluate full-image validation/test with the same sliding-window settings.

## Clone Repo

Set `REPO_BRANCH` to the branch that contains `src/training/train_unet_patch.py` and `src/inference/evaluate_unet_sliding_window.py`.

In [ ]:
REPO_BRANCH = "model/unet-patch384"
!rm -rf /kaggle/working/BTXRD-LViT
!git clone -b {REPO_BRANCH} https://github.com/lehngoc/BTXRD-LViT.git /kaggle/working/BTXRD-LViT

In [ ]:
import json
import shutil
import sys
import zipfile
from pathlib import Path

import pandas as pd
import torch
import yaml

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Locate Dataset

Attach a Kaggle Dataset containing:

```text
data/exports/patches_jitter/patches_384/metadata.csv
data/exports/patches_jitter_val/patches_384/metadata.csv
data/exports/patches_jitter_test/patches_384/metadata.csv
data/exports/btxrd_preprocessed/val.csv
data/exports/btxrd_preprocessed/test.csv
data/processed/images_preprocessed/
data/processed/masks_preprocessed/
```

In [ ]:
REPO_ROOT = Path('/kaggle/working/BTXRD-LViT')
DATA_ROOT = None
target = Path('data/exports/patches_jitter/patches_384/metadata.csv')

for candidate in [Path('/kaggle/input'), *Path('/kaggle/input').glob('**/*')]:
    if candidate.is_dir() and (candidate / target).exists():
        DATA_ROOT = candidate
        break

if DATA_ROOT is None:
    raise FileNotFoundError(f'Could not find {target} under /kaggle/input')

print('REPO_ROOT:', REPO_ROOT)
print('DATA_ROOT:', DATA_ROOT)

In [ ]:
required_paths = [
    DATA_ROOT / 'data/exports/patches_jitter/patches_384/metadata.csv',
    DATA_ROOT / 'data/exports/patches_jitter_val/patches_384/metadata.csv',
    DATA_ROOT / 'data/exports/patches_jitter_test/patches_384/metadata.csv',
    DATA_ROOT / 'data/exports/btxrd_preprocessed/val.csv',
    DATA_ROOT / 'data/exports/btxrd_preprocessed/test.csv',
    DATA_ROOT / 'data/processed/images_preprocessed',
    DATA_ROOT / 'data/processed/masks_preprocessed',
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required paths:\n' + '\n'.join(missing))

for name, path in {
    'train_patch': DATA_ROOT / 'data/exports/patches_jitter/patches_384/metadata.csv',
    'val_patch': DATA_ROOT / 'data/exports/patches_jitter_val/patches_384/metadata.csv',
    'test_patch': DATA_ROOT / 'data/exports/patches_jitter_test/patches_384/metadata.csv',
}.items():
    df = pd.read_csv(path)
    print(name, 'rows=', len(df), 'positive_ratio=', round(float(df['is_positive'].mean()), 4))

## Runtime Config

The config keeps patch training and full-image sliding-window validation in one run. `best.pt` is selected by full-image validation Dice, not patch Dice.

In [ ]:
BASE_CONFIG = REPO_ROOT / 'configs/train_unet_patch384.yaml'
with BASE_CONFIG.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

OUTPUT_DIR = Path('/kaggle/working/experiments/E2_unet_patch384_preprocessed_pos060')
cfg['data']['root_dir'] = str(DATA_ROOT)
cfg['training']['device'] = 'cuda'
cfg['training']['batch_size'] = 4
cfg['training']['num_workers'] = 2
cfg['training']['epochs'] = 100
cfg['training']['early_stopping_patience'] = 20
cfg['training']['full_val_interval'] = 5
cfg['training']['output_dir'] = str(OUTPUT_DIR)
cfg['sliding_window']['patch_size'] = 384
cfg['sliding_window']['stride'] = 192
cfg['sliding_window']['merge'] = 'average_probability'
cfg['sliding_window']['batch_size'] = 4
cfg['metrics']['threshold'] = 0.5

RUNTIME_CONFIG = Path('/kaggle/working/train_unet_patch384_sliding_runtime.yaml')
with RUNTIME_CONFIG.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(RUNTIME_CONFIG.read_text())

## Optional: Restore Previous Checkpoint

If resuming a previous Kaggle run, attach a Dataset or output artifact that contains `last.pt` and optionally `best.pt`. This cell auto-detects the first `last.pt` under `/kaggle/input` and copies checkpoint artifacts into the expected output directory.

In [ ]:
RESUME_FROM_INPUT = False

if RESUME_FROM_INPUT:
    candidates = list(Path('/kaggle/input').glob('**/last.pt'))
    if not candidates:
        raise FileNotFoundError('RESUME_FROM_INPUT=True but no last.pt found under /kaggle/input')
    source_last = candidates[0]
    source_dir = source_last.parent
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name in ['last.pt', 'best.pt', 'history.csv', 'best_summary.json', 'config.json']:
        src = source_dir / name
        if src.exists():
            shutil.copy2(src, OUTPUT_DIR / name)
            print('restored', src, '->', OUTPUT_DIR / name)
else:
    print('Checkpoint restore skipped.')

## Smoke Test

In [ ]:
!python {REPO_ROOT}/src/training/smoke_test_patch_pipeline.py --config {RUNTIME_CONFIG} --samples-per-split 8

## Train Patch UNet + Full-Image Sliding Validation

Use `--auto-resume` so interrupted sessions continue from `OUTPUT_DIR/last.pt` when it exists.

In [ ]:
!python {REPO_ROOT}/src/training/train_unet_patch.py --config {RUNTIME_CONFIG} --device cuda --auto-resume

## Full-Image Sliding-Window Evaluation

This is the main validation/test result. Patch metrics are only diagnostics.

In [ ]:
BEST_CKPT = OUTPUT_DIR / 'best.pt'
LAST_CKPT = OUTPUT_DIR / 'last.pt'
assert BEST_CKPT.exists(), f'Missing best checkpoint: {BEST_CKPT}'
assert LAST_CKPT.exists(), f'Missing last checkpoint: {LAST_CKPT}'

!python {REPO_ROOT}/src/inference/evaluate_unet_sliding_window.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split val --device cuda --output {OUTPUT_DIR}/val_sliding_metrics.json
!python {REPO_ROOT}/src/inference/evaluate_unet_sliding_window.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split test --device cuda --output {OUTPUT_DIR}/test_sliding_metrics.json

In [ ]:
def load_json(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)

for split in ['val', 'test']:
    metrics = load_json(OUTPUT_DIR / f'{split}_sliding_metrics.json')
    print(f'[{split} sliding-window]')
    for key in [
        'tumor_dice',
        'tumor_iou',
        'tumor_precision',
        'tumor_recall',
        'normal_pred_area_ratio',
        'normal_fp_image_rate',
        'avg_windows_per_image',
        'seconds_per_image',
    ]:
        print(f'  {key}: {metrics[key]:.6f}')

## Threshold Sweep on Full-Image Validation

Use this to inspect the Dice / false-positive trade-off without retraining.

In [ ]:
sweep_rows = []
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    out_path = OUTPUT_DIR / f'val_sliding_metrics_thr{int(threshold * 100):02d}.json'
    !python {REPO_ROOT}/src/inference/evaluate_unet_sliding_window.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split val --device cuda --threshold {threshold} --output {out_path}
    metrics = load_json(out_path)
    sweep_rows.append({
        'threshold': threshold,
        'tumor_dice': metrics['tumor_dice'],
        'tumor_iou': metrics['tumor_iou'],
        'normal_fp_image_rate': metrics['normal_fp_image_rate'],
        'normal_pred_area_ratio': metrics['normal_pred_area_ratio'],
    })

with (OUTPUT_DIR / 'threshold_sweep_sliding_metrics.json').open('w', encoding='utf-8') as f:
    json.dump(sweep_rows, f, indent=2)

pd.DataFrame(sweep_rows)

## Package Artifacts

Download this zip or create a Kaggle Dataset from the output to resume later.

In [ ]:
ARTIFACT_ZIP = Path('/kaggle/working/E2_unet_patch384_sliding_window_artifacts.zip')
artifact_names = [
    'best.pt',
    'last.pt',
    'history.csv',
    'best_summary.json',
    'config.json',
    'val_sliding_metrics.json',
    'test_sliding_metrics.json',
    'threshold_sweep_sliding_metrics.json',
]

with zipfile.ZipFile(ARTIFACT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in artifact_names:
        path = OUTPUT_DIR / name
        if path.exists():
            archive.write(path, arcname=name)
    for path in sorted(OUTPUT_DIR.glob('val_sliding_epoch_*.json')):
        archive.write(path, arcname=path.name)

print('Created:', ARTIFACT_ZIP)
print('Files in artifact zip:')
with zipfile.ZipFile(ARTIFACT_ZIP, 'r') as archive:
    print('\n'.join(archive.namelist()))